# Etapa 2: Selección de técnica de muestreo para la construcción de muestra inicial

**Materia:** Análisis de grandes volúmenes de datos (Gpo 10)  
**Institución:** Tecnológico de Monterrey, Posgrados  
**Equipo 12:**

- Carlos Eduardo Vega Campos (A01797803)
- Marco Emilio Jimenez Jimenez (A01797948)
- Martha Alicia Villalobos Facundo (A01840063)
- Jonathan Javier Monsalve Giraldo (A01840272)

**Profesores:** Dr. Iván Olmos Pineda, Luis Daniel Mendoza  
**Fecha:** 17 de mayo de 2026  
**Dataset:** NYC TLC Yellow Taxi Trip Records 2024-2025

## Objetivo del notebook

Construir una muestra representativa M de la población de viajes Yellow Taxi NYC mediante muestreo estratificado con calibración histórica. La Etapa 1 del proyecto caracterizó el dataset; esta etapa parte de los datos crudos, aplica limpieza basada en los hallazgos de Etapa 1, valida D contra distribuciones históricas verificables, y extrae M mediante `sampleBy` con piso mínimo por estrato. El notebook está pensado para ejecutarse de forma portable, tanto localmente como en Google Colab; todas las rutas de datos son relativas al notebook (`./data/raw`).

## 1. Configuración del entorno

Iniciamos sesión local de Spark. Configuración mínima: subir `spark.sql.debug.maxToStringFields` para evitar truncamiento de logs en agregaciones grandes.

Notebook portable: las rutas son relativas. Funciona en Ubuntu VM local y en Google Colab si previamente se instala Java.

In [1]:
# Dependencias de Python para el notebook. Idempotente.
!pip install -q pyspark findspark pandas matplotlib


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
# Solo en Google Colab: descomentar para instalar Java (la JVM que ejecuta Spark).
# Localmente con env-pyspark esta línea no es necesaria.
# !apt-get install openjdk-8-jdk-headless -qq > /dev/null

In [3]:
import findspark
findspark.init()

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, ByteType, ShortType,
    FloatType, TimestampNTZType, StringType,
)
from pathlib import Path
import json

spark = SparkSession.builder.master("local[*]").getOrCreate()

# Sube el umbral del log de planes (default 25) para evitar WARN benignos al agregar
# múltiples columnas en una sola llamada.
spark.conf.set("spark.sql.debug.maxToStringFields", 100)

print(f"Spark versión: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/17 12:52:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark versión: 4.1.1


### 1.2 Descarga reproducible de los datos

Reusamos el patrón de Etapa 1: descarga idempotente desde el CDN público de TLC a `./data/raw/` solo si el archivo no existe. Esto mantiene el notebook autosuficiente y portable. En una máquina donde ya se descargaron los 24 parquets mensuales más el catálogo de zonas (por ejemplo, tras ejecutar el notebook de Etapa 1), todas las descargas devuelven `skip` y la celda termina en segundos.

In [4]:
import subprocess

CDN_BASE = "https://d37ci6vzurychx.cloudfront.net"
DATA_DIR = Path("data/raw")
YEARS = [2024, 2025]
LOOKUP_FILE = "taxi_zone_lookup.csv"


def download_if_missing(download_url, target_path):
    """Descarga `download_url` a `target_path` solo si `target_path` no existe.

    Devuelve un string con el estado: 'skip', 'ok' o 'error: <mensaje>'.
    """
    if target_path.exists():
        return "skip"

    target_path.parent.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ["curl", "-sSL", "-o", str(target_path), download_url],
        capture_output=True,
        timeout=900,
    )

    if result.returncode != 0:
        return f"error: curl exit {result.returncode}"

    return "ok"

In [5]:
# Parquets mensuales de viajes
for year in YEARS:
    for month in range(1, 13):
        filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
        url = f"{CDN_BASE}/trip-data/{filename}"
        target = DATA_DIR / filename
        status = download_if_missing(url, target)
        print(f"{status:>6}  {filename}")

# Tabla de referencia de zonas de taxi
url = f"{CDN_BASE}/misc/{LOOKUP_FILE}"
target = DATA_DIR / LOOKUP_FILE
status = download_if_missing(url, target)
print(f"{status:>6}  {LOOKUP_FILE}")

    ok  yellow_tripdata_2024-01.parquet
    ok  yellow_tripdata_2024-02.parquet
    ok  yellow_tripdata_2024-03.parquet
    ok  yellow_tripdata_2024-04.parquet
    ok  yellow_tripdata_2024-05.parquet
    ok  yellow_tripdata_2024-06.parquet
    ok  yellow_tripdata_2024-07.parquet
    ok  yellow_tripdata_2024-08.parquet
    ok  yellow_tripdata_2024-09.parquet
    ok  yellow_tripdata_2024-10.parquet
    ok  yellow_tripdata_2024-11.parquet
    ok  yellow_tripdata_2024-12.parquet
    ok  yellow_tripdata_2025-01.parquet
    ok  yellow_tripdata_2025-02.parquet
    ok  yellow_tripdata_2025-03.parquet
    ok  yellow_tripdata_2025-04.parquet
    ok  yellow_tripdata_2025-05.parquet
    ok  yellow_tripdata_2025-06.parquet
    ok  yellow_tripdata_2025-07.parquet
    ok  yellow_tripdata_2025-08.parquet
    ok  yellow_tripdata_2025-09.parquet
    ok  yellow_tripdata_2025-10.parquet
    ok  yellow_tripdata_2025-11.parquet
    ok  yellow_tripdata_2025-12.parquet
    ok  taxi_zone_lookup.csv


In [6]:
files = sorted(DATA_DIR.glob("*"))
total_bytes = sum(f.stat().st_size for f in files)

for f in files:
    size_mb = f.stat().st_size / (1024 ** 2)
    print(f"{size_mb:>8.1f} MB   {f.name}")

print()
print(f"Archivos: {len(files)} (esperados: {len(YEARS) * 12 + 1})")
print(f"Tamaño total: {total_bytes / (1024 ** 3):.2f} GB")

     0.0 MB   taxi_zone_lookup.csv
    47.6 MB   yellow_tripdata_2024-01.parquet
    48.0 MB   yellow_tripdata_2024-02.parquet
    57.3 MB   yellow_tripdata_2024-03.parquet
    56.4 MB   yellow_tripdata_2024-04.parquet
    59.7 MB   yellow_tripdata_2024-05.parquet
    57.1 MB   yellow_tripdata_2024-06.parquet
    49.9 MB   yellow_tripdata_2024-07.parquet
    48.7 MB   yellow_tripdata_2024-08.parquet
    58.3 MB   yellow_tripdata_2024-09.parquet
    61.4 MB   yellow_tripdata_2024-10.parquet
    57.8 MB   yellow_tripdata_2024-11.parquet
    58.7 MB   yellow_tripdata_2024-12.parquet
    56.4 MB   yellow_tripdata_2025-01.parquet
    57.5 MB   yellow_tripdata_2025-02.parquet
    66.7 MB   yellow_tripdata_2025-03.parquet
    64.2 MB   yellow_tripdata_2025-04.parquet
    74.2 MB   yellow_tripdata_2025-05.parquet
    70.1 MB   yellow_tripdata_2025-06.parquet
    63.8 MB   yellow_tripdata_2025-07.parquet
    59.4 MB   yellow_tripdata_2025-08.parquet
    69.1 MB   yellow_tripdata_2025-09.parquet